# 第7章: 機械学習

本章では、[Stanford Sentiment Treebank (SST)](https://nlp.stanford.edu/sentiment/) データセットを用い、評判分析器（ポジネガ分類器）を構築する。ここでは処理を簡略化するため、[General Language Understanding Evaluation (GLUE)](https://gluebenchmark.com/) ベンチマークで配布されているSSTデータセットを用いる。


In [62]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 60. データの入手・整形

GLUEのウェブサイトから[SST-2](https://dl.fbaipublicfiles.com/glue/data/SST-2.zip)データセットを取得せよ。学習データ（`train.tsv`）と検証データ（`dev.tsv`）のぞれぞれについて、ポジティブ (1) とネガティブ (0) の事例数をカウントせよ。

In [63]:
import pandas as pd

In [64]:
train_data = pd.read_csv('/content/drive/MyDrive/nlp100/lesson07/SST-2/train.tsv', sep = '\t') # tsvファイルを読み込むときは、read_csvのsepを'\t'にする
train_posi = train_data[train_data['label'] == 1]
train_nega = train_data[train_data['label'] == 0]
print(f'Train: ポジティブの事例数: {len(train_posi)}, ネガティブの事例数: {len(train_nega)}, 合計事例数: {len(train_data)}')

valid_data = pd.read_csv('/content/drive/MyDrive/nlp100/lesson07/SST-2/dev.tsv', sep = '\t')
valid_posi = valid_data[valid_data['label'] == 1]
valid_nega = valid_data[valid_data['label'] == 0]
print(f'Train: ポジティブの事例数: {len(valid_posi)}, ネガティブの事例数: {len(valid_nega)}, 合計事例数: {len(valid_data)}')

Train: ポジティブの事例数: 37569, ネガティブの事例数: 29780, 合計事例数: 67349
Train: ポジティブの事例数: 444, ネガティブの事例数: 428, 合計事例数: 872


## 61. 特徴ベクトル

Bag of Words (BoW) に基づき、学習データ（`train.tsv`）および検証データ（`dev.tsv`）のテキストを特徴ベクトルに変換したい。ここで、ある事例のテキストの特徴ベクトルは、テキスト中に含まれる単語（スペース区切りのトークン）の出現頻度で構成する。例えば、"too loud , too goofy"というテキストに対応する特徴ベクトルは、以下のような辞書オブジェクトで表現される。

```python
{'too': 2, 'loud': 1, ',': 1, 'goofy': 1}
```

各事例はテキスト、特徴ベクトル、ラベルを格納した辞書オブジェクトでまとめておく。例えば、先ほどの"too loud , too goofy"に対してラベル"0"（ネガティブ）が付与された事例は、以下のオブジェクトで表現される。

```python
{'text': 'too loud , too goofy', 'label': '0', 'feature': {'too': 2, 'loud': 1, ',': 1, 'goofy': 1}}
```

学習データと検証データの各事例を上記のような辞書オブジェクトに変換したうえで、学習データと検証データのそれぞれを、辞書オブジェクトのリストとして表現せよ。さらに、学習データの最初の事例について、正しく特徴ベクトルに変換できたか、目視で確認せよ。

In [65]:
from sklearn.feature_extraction.text import CountVectorizer

In [66]:
train_data[0:1]

,sentence,label
0,hide new secretions from the parental units,0


In [67]:
vectorizer = CountVectorizer()
bow_matrix = vectorizer.fit_transform(train_data['sentence'].to_list())
feature_names = vectorizer.get_feature_names_out()

In [68]:
head = list(bow_matrix[0].toarray()[0])
for i in range(len(head)):
  if head[i] != 0:
    print(f'{feature_names[i]}: {head[i]}')

from: 1
hide: 1
new: 1
parental: 1
secretions: 1
the: 1
units: 1


In [69]:
valid_data[0:1]

,sentence,label
0,it 's a charming and often affecting journey .,1


In [70]:
bow_matrix = vectorizer.transform(valid_data['sentence'].to_list())
feature_names = vectorizer.get_feature_names_out()

In [71]:
head = list(bow_matrix[0].toarray()[0])
for i in range(len(head)):
  if head[i] != 0:
    print(f'{feature_names[i]}: {head[i]}')

affecting: 1
and: 1
charming: 1
it: 1
journey: 1
often: 1


## 62. 学習

61で構築した学習データの特徴ベクトルを用いて、ロジスティック回帰モデルを学習せよ。

In [72]:
from sklearn.linear_model import LogisticRegression

In [73]:
X_train = vectorizer.transform(train_data['sentence'].to_list()).toarray()
y_train = train_data['label'].to_numpy()
print(f'X: {len(X_train), type(X_train)}, y: {len(y_train), type(y_train)}')

X: (67349, <class 'numpy.ndarray'>), y: (67349, <class 'numpy.ndarray'>)


In [74]:
classifier = LogisticRegression(random_state = 42)
classifier.fit(X_train, y_train)

LogisticRegression(random_state=42)

## 63. 予測

学習したロジスティック回帰モデルを用い、検証データの先頭の事例のラベル（ポジネガ）を予測せよ。また、予測されたラベルが検証データで付与されていたラベルと一致しているか、確認せよ。

In [75]:
X_valid = vectorizer.transform(valid_data['sentence'].to_list()).toarray()

In [76]:
classifier.predict([X_valid[0]])

array([1])

## 64. 条件付き確率

学習したロジスティック回帰モデルを用い、検証データの先頭の事例を各ラベル（ポジネガ）に分類するときの条件付き確率を求めよ。

In [77]:
classifier.predict_proba([X_valid[0]])

array([[0.00241462, 0.99758538]])

## 65. テキストのポジネガの予測

与えられたテキストのポジネガを予測するプログラムを実装せよ。例えば、テキストとして"the worst movie I 've ever seen"を与え、ロジスティック回帰モデルの予測結果を確認せよ。


In [78]:
texts = [
    "the worst movie I 've ever seen"
]
X_text = vectorizer.transform(texts).toarray()
classifier.predict([X_valid[0]])

array([1])

## 66. 混同行列の作成

学習したロジスティック回帰モデルの検証データにおける混同行列（confusion matrix）を求めよ。

In [79]:
from sklearn.metrics import confusion_matrix

In [80]:
y_valid = valid_data['label'].to_numpy()
y_pred = classifier.predict(X_valid)

In [81]:
confusion_matrix(y_valid, y_pred) # 行が正解、列が予測  (0,0): 真陰性, (0,1): 偽陽性, (1,0): 偽陰性, (1,1): 真陽性

array([[332,  96],
       [ 62, 382]])

## 67. 精度の計測

学習したロジスティック回帰モデルの正解率、適合率、再現率、F1スコアを、学習データおよび検証データ上で計測せよ。

In [82]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

In [83]:
# 学習データにおける精度
y_pred = classifier.predict(X_train)
print(f'正解率: {accuracy_score(y_train, y_pred)}')
print(f'適合率: {precision_score(y_train, y_pred)}')
print(f'再現率: {recall_score(y_train, y_pred)}')
print(f'F1スコア: {f1_score(y_train, y_pred)}')

正解率: 0.936791934549882
適合率: 0.9374179316140554
再現率: 0.95011844872102
F1スコア: 0.9437254616838738


In [84]:
# 検証データにおける精度
y_pred = classifier.predict(X_valid)
print(f'正解率: {accuracy_score(y_valid, y_pred)}')
print(f'適合率: {precision_score(y_valid, y_pred)}')
print(f'再現率: {recall_score(y_valid, y_pred)}')
print(f'F1スコア: {f1_score(y_valid, y_pred)}')

正解率: 0.8188073394495413
適合率: 0.799163179916318
再現率: 0.8603603603603603
F1スコア: 0.8286334056399133


## 68. 特徴量の重みの確認

学習したロジスティック回帰モデルの中で、重みの高い特徴量トップ20と、重みの低い特徴量トップ20を確認せよ。

In [85]:
def Myabs(value):
  return abs(value)

In [86]:
weights = classifier.coef_.tolist()[0]
features_data = {'word': [], 'weight': []}
for name, weight in zip(feature_names, weights):
  features_data['word'].append(name)
  features_data['weight'].append(weight)

features = pd.DataFrame(features_data)
features['weight'] = features['weight'].apply(Myabs) # 重みが低い = 0に近い
features = features.sort_values('weight', ascending = False)

In [87]:
features.head(20)

,word,weight
6905,lacking,4.398807
13647,worst,4.206078
6907,lacks,4.177701
4456,failure,3.745587
3306,devoid,3.617297
11744,stupid,3.572183
7687,mess,3.548878
9958,remarkable,3.457285
9876,refreshing,3.388322
1456,bore,3.338057


In [88]:
features.tail(20)

,word,weight
3853,edition,0.001629
2078,chin,0.001538
12821,underscore,0.001500
9225,powered,0.001467
9542,punk,0.001458
5184,gets,0.001392
6867,korea,0.001378
11892,supply,0.001345
2533,conjuring,0.001201
9470,provide,0.001040


## 69. 正則化パラメータの変更

ロジスティック回帰モデルを学習するとき、正則化の係数（ハイパーパラメータ）を調整することで、学習時の適合度合いを制御できる。正則化の係数を変化させながらロジスティック回帰モデルを学習し、検証データ上の正解率を求めよ。実験の結果は、正則化パラメータを横軸、正解率を縦軸としたグラフにまとめよ。

In [89]:
import matplotlib.pyplot as plt

In [90]:
C = np.linspace(0.001, 1, 5).tolist()
accuracies = []
for c in C:
  # 学習
  classifier = LogisticRegression(penalty = 'l2', C = c, random_state = 42) # penaltyは正則化の方法, Cは正則化の強さ(正の値)
  classifier.fit(X_train, y_train)

  # 予測
  y_pred = classifier.predict(X_valid)

  # 正解率検証
  accuracies.append(accuracy_score(y_valid, y_pred))

In [91]:
plt.plot(pd.Series(C), pd.Series(accuracies))
plt.show()

AttributeError: module 'pandas' has no attribute 'Serise'